# E-Commerce Support Multi-Agent Workflow

Colab-ready assignment: Catalog and Order Support are distinct LLM agents with role prompts and least-privilege tools. Critical order changes pause for human review before final output.

In [ ]:
!pip -q install langchain-openai langgraph

import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")
os.environ.setdefault("OPENAI_MODEL", "gpt-4o-mini")

In [ ]:
from typing import Annotated, Literal, TypedDict
from uuid import uuid4
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt

class WorkflowState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    user_request: str
    intent: Literal["catalog", "order", "action", "unsupported"]
    final_answer: str
    human_decision: str
    human_feedback: str

CATALOG = [{"id": 1, "name": "Atlas Laptop", "price": 1200, "stock": 3}, {"id": 2, "name": "Nova Phone", "price": 750, "stock": 7}]
ORDERS = [{"id": 42, "status": "PREPARING", "item": "Atlas Laptop"}]

@tool
def search_products(query: str) -> list[dict]:
    """Search the active product catalog by customer request."""
    words = set(query.lower().split())
    return [product for product in CATALOG if words & set(product["name"].lower().split())]

@tool
def get_product(product_id: int) -> dict:
    """Get authoritative details for one product ID."""
    return next((product for product in CATALOG if product["id"] == product_id), {})

@tool
def get_my_orders() -> list[dict]:
    """Get the signed-in demonstration customer's order history."""
    return ORDERS

In [ ]:
CATALOG_PROMPT = """You are the Catalog Agent. Use catalog tools before answering. Cite only returned fields; never invent price or stock."""
ORDER_PROMPT = """You are the Order Support Agent. Use the order tool before answering. Never mutate or disclose another customer's data."""
catalog_model = ChatOpenAI(model=os.environ["OPENAI_MODEL"], temperature=0).bind_tools([search_products, get_product])
order_model = ChatOpenAI(model=os.environ["OPENAI_MODEL"], temperature=0).bind_tools([get_my_orders])

def router(state: WorkflowState):
    request = state["user_request"].lower()
    if any(term in request for term in ("cancel", "refund", "checkout")) or ("change" in request and "address" in request):
        return {"intent": "action"}
    if any(term in request for term in ("order", "delivery", "shipping", "track")):
        return {"intent": "order"}
    if any(term in request for term in ("product", "laptop", "phone", "price", "recommend")):
        return {"intent": "catalog"}
    return {"intent": "unsupported"}

def catalog_agent(state: WorkflowState):
    return {"messages": [catalog_model.invoke([SystemMessage(content=CATALOG_PROMPT), *state["messages"]])]}

def order_agent(state: WorkflowState):
    return {"messages": [order_model.invoke([SystemMessage(content=ORDER_PROMPT), *state["messages"]])]}

def approval_gate(_state: WorkflowState):
    decision = interrupt({"question": "Approve, reject, or revise this action?"})
    return {"human_decision": decision["decision"], "human_feedback": decision.get("feedback")}

def finalizer(state: WorkflowState):
    if state["intent"] == "action":
        answer = f"Action {state.get('human_decision', 'rejected')}. {state.get('human_feedback') or ''}".strip()
    elif state["intent"] == "unsupported":
        answer = "I can help with products or your order status."
    else:
        answer = state["messages"][-1].content
    return {"final_answer": answer, "messages": [AIMessage(content=answer)]}

In [ ]:
def after_router(state: WorkflowState):
    return {"catalog": "catalog_agent", "order": "order_agent", "action": "approval"}.get(state["intent"], "finalizer")

def after_agent(state: WorkflowState):
    return "tools" if state["messages"][-1].tool_calls else "finalizer"

def build_graph():
    graph = StateGraph(WorkflowState)
    graph.add_node("router", router)
    graph.add_node("catalog_agent", catalog_agent)
    graph.add_node("catalog_tools", ToolNode([search_products, get_product]))
    graph.add_node("order_agent", order_agent)
    graph.add_node("order_tools", ToolNode([get_my_orders]))
    graph.add_node("approval", approval_gate)
    graph.add_node("finalizer", finalizer)
    graph.add_edge(START, "router")
    graph.add_conditional_edges("router", after_router)
    graph.add_conditional_edges("catalog_agent", after_agent, {"tools": "catalog_tools", "finalizer": "finalizer"})
    graph.add_edge("catalog_tools", "catalog_agent")
    graph.add_conditional_edges("order_agent", after_agent, {"tools": "order_tools", "finalizer": "finalizer"})
    graph.add_edge("order_tools", "order_agent")
    graph.add_edge("approval", "finalizer")
    graph.add_edge("finalizer", END)
    return graph.compile(checkpointer=MemorySaver())

workflow = build_graph()

def execute_workflow(user_request: str):
    """Required assignment entry point: start a checkpointed workflow."""
    session_id = str(uuid4())
    config = {"configurable": {"thread_id": session_id}}
    state = workflow.invoke({"user_request": user_request, "messages": [HumanMessage(content=user_request)]}, config)
    return {"session_id": session_id, "state": state}

def resume_workflow(session_id: str, decision: str, feedback: str = ""):
    return workflow.invoke(Command(resume={"decision": decision, "feedback": feedback}), {"configurable": {"thread_id": session_id}})

In [ ]:
# Five required scenarios: three tool workflows plus rejected and revised HITL requests.
cases = [
    ("Recommend a laptop", None),
    ("Show product 1 details", None),
    ("Where is my order?", None),
    ("Cancel my order", {"decision": "rejected", "feedback": "Keep the order active."}),
    ("Change my delivery address", {"decision": "revise", "feedback": "Ask for the new address first."}),
]

for request, review in cases:
    result = execute_workflow(request)
    state = result["state"]
    if review:
        assert "__interrupt__" in state, "Critical actions must interrupt"
        state = resume_workflow(result["session_id"], **review)
    print(f"{request}: {state['final_answer']}")